# Solution: Training a Neural Forecasting Model

A streaming service needs to forecast monthly subscribers. Growth follows an S-curve that ARIMA struggles with. Train an N-BEATS model with quantile regression and compare it to an ARIMA baseline.

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import matplotlib as mplimport matplotlib.pyplot as plt# Udacity brand colorsUB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"}UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62", "Medium Gray": "#9C9FA8", "Gray": "#CECFD4", "Light Gray": "#F2F2F2", "White": "#FFFFFF"}US = {"Orange": "#EE7622", "Yellow": "#F9DC5C", "Red": "#9C0D22", "Green": "#23CE6B"}# Thicker lines for visibilitympl.rcParams["lines.linewidth"] = 3mpl.rcParams["axes.linewidth"] = 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape
from darts.utils.likelihood_models import QuantileRegression

HORIZON = 12

In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date").asfreq("MS")
subscribers = df["subscribers"]
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))

print(f"Data: {len(subscribers)} months, {subscribers.index[0].date()} to {subscribers.index[-1].date()}")
print(f"Train: {len(train)} months, Test: {len(test)} months")

## Task 1: Train N-BEATS

Create an `NBEATSModel` with `input_chunk_length=24`, `output_chunk_length=12`, `QuantileRegression` likelihood (quantiles: 0.05, 0.25, 0.5, 0.75, 0.95), `random_state=42`. Train for the given number of epochs. Return the fitted model.

In [ ]:
def train_nbeats(train, input_length=24, output_length=12, epochs=50):
    model = NBEATSModel(
        input_chunk_length=input_length,
        output_chunk_length=output_length,
        n_epochs=epochs,
        likelihood=QuantileRegression(quantiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
        random_state=42,
        pl_trainer_kwargs={"enable_progress_bar": True},
    )
    model.fit(train)
    return model

nbeats = train_nbeats(train, epochs=50)

In [ ]:
nbeats_fc = nbeats.predict(HORIZON, num_samples=100)

In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
TimeSeries.from_series(subscribers).plot(ax=ax, label="Actual", color="black", linewidth=1.5)
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color="tab:blue")
nbeats_fc.plot(ax=ax, label="N-BEATS", color="tab:orange", low_quantile=0.05, high_quantile=0.95)
ax.set_ylabel("Subscribers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
print("=== Forecast at Month 12 ===")
print(f"  ARIMA: {arima_fc.values().flatten()[-1]:,.0f}")
print(f"  N-BEATS: {nbeats_fc.values().flatten()[-1]:,.0f}")